# [Cell 0] Colab Full-Dataset CFM Training Guide

## Recommended flow (Stage 2 not required)

Run in this order:

| Step | Cell | Purpose | Time |
|------|------|---------|------|
| 1 | **1** | Set project directory + `sys.path` | < 1 min |
| 2 | **2** | Install dependencies (`requirements.txt` + transformers/accelerate) | 2–5 min |
| 3 | **3** | WandB login (optional) | 1 min |
| 4 | **4** | Check required data files | < 1 min |
| 5 | **8** | Prepare extractor bundle (`best.ckpt` + `args.json`) | < 1 min |
| 6 | **9** | Resolve `extractor_path` | < 1 min |
| 7 | **12** | Stage 3 CFM full training | 45–180 min |
| 8 | **13** | Visualize 4 DI-spread GT vs inference + PSNR/SSIM/FID | 2–5 min |
| 9 | **14** | Grounding DINO + loaded DINOv3 interference-masked DI evaluation | 2–5 min |
| 10 | **15** | Standalone GT visual dashboard: one random image per column | 2–10 min |

### Standalone DI-MLP training and dashboard (no CFM checkpoint required)
Run only: **1 → 2 → 4 → 15**. Cell 15 loads frozen Grounding DINO + DINOv3, caches masked DINOv3 features from GT train/val images, trains the DI MLP on the train split, reports validation MAE/RMSE/R², saves the MLP checkpoint, then visualizes the GT validation dashboard. The initial feature-cache pass may take substantially longer than MLP training; later reruns reuse `/content/di_mlp_dinov3_features.pt`.

Optional debug/smoke cells: **5, 6, 7, 10, 11**.

## CFM condition
`c = [hidden_state, direction, DI]`
- `hidden_state`: from Stage 1 GRU extractor (`best.ckpt`)
- `direction`, `DI`: from `Data/DamagePattern/label.csv`

## Note about Cell 8
If `args.json` is missing, Cell 8 reads `hyper_parameters` directly from `best.ckpt` and writes a compatible `args.json`.

## Inference snapshots during training
CFM saves/logs inference images every 10 epochs by default:
- `results/.../inference_samples/epoch_XXXXX.png`


In [ ]:
# [Cell 1] Set project directory + Python path
PROJECT_DIR = "/content/intership-project"

import os, sys
assert os.path.isdir(PROJECT_DIR), f"Project dir not found: {PROJECT_DIR}"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
%cd /content/intership-project
print("Project dir:", PROJECT_DIR)
print("sys.path[0]:", sys.path[0])


In [ ]:
# [Cell 2] Install dependencies
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q transformers accelerate


In [ ]:
# [Cell 3] Optional WandB login
USE_WANDB = True  # set False to disable

if USE_WANDB:
    import wandb
    wandb.login()
    WANDB_FLAG = "--use_wandb"
else:
    WANDB_FLAG = ""

print("WANDB_FLAG:", WANDB_FLAG)


In [ ]:
# [Cell 4] Check required data files
required_paths = [
    "Data/Hysteresis/info_w_mander_big.csv",
    "Data/Hysteresis/id_list_small.json",
    "Data/DamagePattern/label.csv",
    "Data/DamagePattern/id_list.json",
]

for p in required_paths:
    print(p, "OK" if os.path.exists(p) else "MISSING")


In [ ]:
# [Cell 5] Optional one-click runner
MODE = "smoke"      # "smoke" or "full"
BACKEND = "cfm"     # "gan" or "cfm"

wandb_opt = "--with-wandb" if USE_WANDB else ""
!bash scripts/colab_quickstart.sh {wandb_opt} --mode={MODE} --backend={BACKEND}


In [ ]:
# [Cell 6] Stage 1 smoke train (optional)
!python train_hysteresis.py --architecture HystereticGRU --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# [Cell 7] Stage 2 smoke train (optional)
!python train_quantification.py --backbone vit_b_16 --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# [Cell 8] Prepare extractor bundle for Stage 3
# - If best.ckpt is in project root, copy it to phase1_extractor/
# - If args.json is missing, generate from ckpt hyper_parameters

import os, json, shutil, torch

PROJECT_ROOT = "/content/intership-project"
EXTRACTOR_DIR = os.path.join(PROJECT_ROOT, "phase1_extractor")
os.makedirs(EXTRACTOR_DIR, exist_ok=True)

ckpt_candidates = [
    os.path.join(EXTRACTOR_DIR, "best.ckpt"),
    os.path.join(PROJECT_ROOT, "best.ckpt"),
]
ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), None)
assert ckpt_path is not None, "No best.ckpt found (expected in project root or phase1_extractor/)"

target_ckpt = os.path.join(EXTRACTOR_DIR, "best.ckpt")
if os.path.abspath(ckpt_path) != os.path.abspath(target_ckpt):
    shutil.copy2(ckpt_path, target_ckpt)
    ckpt_path = target_ckpt
    print("Copied best.ckpt ->", target_ckpt)
else:
    print("Using best.ckpt ->", ckpt_path)

args_path = os.path.join(EXTRACTOR_DIR, "args.json")
if not os.path.exists(args_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    hp = ckpt.get("hyper_parameters") or {}
    assert hp, "Checkpoint has no hyper_parameters; please provide matching args.json"

    # Safe defaults aligned with train_hysteresis.py and train_GRU.sh
    args_payload = {
        "architecture": "HystereticGRU",
        "hidden_state_dim": 512,
        "layers": 4,
        "MLP_activation": "softsign",
        "bidirectional": True,
        "simulate": True,
        "scale": True,
        "data_size": "big",
        "lr": 1e-4,
        "epoch": 5000,
        "batch_size": 8,
        "num_workers": 2,
        "maximum": 1.0,
        "data_type": {},
        "data_version": 1,
    }
    args_payload.update(hp)

    # Infer signal_feat if missing
    sd = ckpt["state_dict"]
    if "signal_feat" not in args_payload:
        if "L.gru.weight_ih_l0" in sd:
            args_payload["signal_feat"] = int(sd["L.gru.weight_ih_l0"].shape[1])
        elif "L.lstm.weight_ih_l0" in sd:
            args_payload["signal_feat"] = int(sd["L.lstm.weight_ih_l0"].shape[1])
        else:
            raise RuntimeError("Cannot infer signal_feat from checkpoint weights")

    with open(args_path, "w", encoding="utf-8") as f:
        json.dump(args_payload, f, indent=2)
    print("Generated args.json ->", args_path)
else:
    print("Using existing args.json ->", args_path)

print("Extractor bundle ready:", EXTRACTOR_DIR)


In [ ]:
# [Cell 9] Resolve extractor_path
import glob, os

# Preferred manual directory for your imported Phase-1 checkpoint bundle.
# Put both files inside this directory (or its subfolders): args.json + best.ckpt
MANUAL_EXTRACTOR_PATH = "/content/intership-project/phase1_extractor"

# Extra auto-search candidates (first valid one will be used)
AUTO_CANDIDATES = [
    MANUAL_EXTRACTOR_PATH,
    "/content/intership-project",
    "/content/phase1_extractor",
]

# Fallback author path (if exists)
AUTHOR_EXTRACTOR = "results/HystereticPrediction/HystereticGRU/2024-06-12_18-13-47"


def latest_run(base_glob):
    runs = sorted(glob.glob(base_glob), key=os.path.getmtime)
    assert runs, f"No runs found for: {base_glob}. Run Cell 6 first."
    return runs[-1]


def has_required_extractor_files(run_dir):
    if not os.path.isdir(run_dir):
        return False
    has_args = len(glob.glob(os.path.join(run_dir, "**", "args.json"), recursive=True)) > 0
    has_ckpt = len(glob.glob(os.path.join(run_dir, "**", "best.ckpt"), recursive=True)) > 0
    return has_args and has_ckpt


extractor_path = None
for cand in AUTO_CANDIDATES:
    if has_required_extractor_files(cand):
        extractor_path = cand
        print("Using manual/auto extractor:", extractor_path)
        break

if extractor_path is None and has_required_extractor_files(AUTHOR_EXTRACTOR):
    extractor_path = AUTHOR_EXTRACTOR
    print("Using author extractor:", extractor_path)

if extractor_path is None:
    extractor_path = latest_run("results/HystereticPrediction/HystereticGRU/*")
    print("Using latest local Stage 1 run:", extractor_path)

assert has_required_extractor_files(extractor_path), (
    "extractor_path must contain BOTH args.json and best.ckpt. "
    "If you only copied best.ckpt, also copy the matching args.json from the same Phase-1 run."
)


In [ ]:
# [Cell 10] Stage 3 GAN smoke (optional)
!python train_mixed.py \
  --architecture HystereticCGAN \
  --loss_type SGAN \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning False \
  --direction_conditioning False \
  --add_DI_loss False \
  --image_size 128 \
  --z_dim 100 \
  --G_h_size 128 \
  --D_h_size 128 \
  --SELU False \
  --spectral_D False \
  --no_batch_norm False \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# [Cell 11] Stage 3 CFM smoke (optional)
!python train_mixed.py \
  --architecture CFMADM \
  --cfm_solver heun \
  --cfm_ode_steps 20 \
  --cfm_model_channels 128 \
  --cfm_num_res_blocks 2 \
  --cfm_cond_dim 512 \
  --cfm_dropout 0.0 \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning True \
  --direction_conditioning True \
  --include_di_conditioning True \
  --add_DI_loss False \
  --image_size 128 \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# [Cell 12] Stage 3 CFM full training (packaged)
import os, subprocess, shlex, sys

# Set True on main branch for SD3 VAE latent-space CFM (V4)
USE_LATENT_CFM = True

TRAIN_CFG = {
    "architecture": "CFMADM",
    "cfm_solver": "heun",
    "cfm_ode_steps": 50,
    "extractor_path": extractor_path,
    "data_path": "id_list.json",
    "latent_conditioning": True,
    "direction_conditioning": True,
    "include_di_conditioning": True,
    "add_DI_loss": False,
    "image_size": 128,
    "lr_schedule": "warmup_cosine",
    "warmup_epochs": 10,
    "min_lr_ratio": 0.1,
    "val_every_n_epoch": 10,
    "inference_every_n_epochs": 10,
    "inference_num_images": 6,
    "epoch": 100,
    "batch_size": 4,
    "num_workers": 0,
}

if USE_LATENT_CFM:
    TRAIN_CFG.update({
        "use_sd3_vae_latent": True,
        "sd3_vae_model_id": "stabilityai/stable-diffusion-3-medium-diffusers",
        "sd3_vae_subfolder": "vae",
        "sd3_vae_use_fp16": True,
    })

cmd = ["python", "train_mixed.py"]
for k, v in TRAIN_CFG.items():
    cmd += [f"--{k}", str(v)]
if WANDB_FLAG:
    cmd.append(WANDB_FLAG)

print("Training command:\n", " ".join(shlex.quote(x) for x in cmd))
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
result = subprocess.run(cmd, check=False, env=env)
if result.returncode != 0:
    raise RuntimeError(f"train_mixed.py failed with exit code {result.returncode}. Scroll up for traceback.")


In [ ]:
# [Cell 13] Visualize 4 DI-spread GT vs CFM inference + metrics
import os, glob, json, argparse, importlib
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import transforms
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from src.dataset import PatternLoopDataset

# DI targets: 3 spread in [0,1] + 1 near 1.0
TARGET_DIS = [0.09, 0.43, 0.79, 0.98]


def first_match(pattern):
    matches = glob.glob(pattern, recursive=True)
    assert matches, f"No file found: {pattern}"
    return matches[0]


def latest_cfm_run(base_glob="results/DamagePatternGeneration/*/*"):
    run_dirs = sorted(glob.glob(base_glob), key=os.path.getmtime, reverse=True)
    for run_dir in run_dirs:
        args_candidates = glob.glob(os.path.join(run_dir, "**", "args.json"), recursive=True)
        if not args_candidates:
            continue
        args_data = json.load(open(args_candidates[0], "r"))
        if args_data.get("architecture") == "CFMADM":
            return run_dir
    raise RuntimeError("No CFM run found under results/DamagePatternGeneration/*/*")


def select_di_spread_indices(label_df, target_dis):
    pool = label_df.reset_index(drop=True).copy()
    used_positions = set()
    selected = []
    for t in target_dis:
        cand = pool[~pool.index.isin(used_positions)].copy()
        cand["dist"] = (cand["DI"] - float(t)).abs()
        pick_pos = int(cand.sort_values(["dist", "DI"]).index[0])
        used_positions.add(pick_pos)
        row = pool.iloc[pick_pos]
        selected.append({
            "idx": pick_pos,
            "target_di": float(t),
            "actual_di": float(row["DI"]),
            "filename": row["filename"],
        })
    return selected


def decode_prediction(model, pred_tensor):
    if getattr(model, "use_sd3_vae_latent", False) and getattr(model, "vae", None) is not None:
        return model._decode_from_target_space(pred_tensor)
    return pred_tensor


# 1) Resolve latest CFM run folder
run_dir = latest_cfm_run()
print("CFM run_dir:", run_dir)

# 2) Load saved args and checkpoint
args_path = first_match(os.path.join(run_dir, "**", "args.json"))
run_args = argparse.Namespace(**json.load(open(args_path, "r")))

ckpt_candidates = []
for patt in ["best_fid_*.ckpt", "best_ssim_*.ckpt", "*.ckpt"]:
    ckpt_candidates.extend(glob.glob(os.path.join(run_dir, "**", patt), recursive=True))
assert ckpt_candidates, f"No checkpoint found under {run_dir}"
ckpt_path = sorted(set(ckpt_candidates), key=os.path.getmtime)[-1]
print("Using checkpoint:", ckpt_path)

# 3) Load extractor
extractor_args_path = first_match(os.path.join(run_args.extractor_path, "**", "args.json"))
extractor_args = argparse.Namespace(**json.load(open(extractor_args_path, "r")))
extractor = importlib.import_module("src.model.HystereticPrediction").__dict__[extractor_args.architecture](**vars(extractor_args))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
extractor = extractor.to(device)
extractor_ckpt = first_match(os.path.join(run_args.extractor_path, "**", "best.ckpt"))
extractor.load_state_dict(torch.load(extractor_ckpt, map_location=device, weights_only=False)["state_dict"])
extractor.eval()

# 4) Build validation dataset and pick 4 DI-spread samples
transform = transforms.Compose([
    transforms.Resize((run_args.image_size, run_args.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

val_ds = PatternLoopDataset(
    root="./Data",
    mode="val",
    transform=transform,
    extractor=extractor,
    extractor_args=extractor_args,
    **vars(run_args),
)

selected = select_di_spread_indices(val_ds.label, TARGET_DIS)
print("Selected DI samples:")
for s in selected:
    print(f"  idx={s['idx']} target={s['target_di']:.2f} actual={s['actual_di']:.4f} file={s['filename']}")

# 5) Load CFM model
model = importlib.import_module("src.model.ImageGeneration").__dict__[run_args.architecture](regressor=None, **vars(run_args))
model = model.to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False)["state_dict"], strict=False)
model.eval()

# 6) Run inference for selected samples
records = []
for s in selected:
    real_data, hidden_state, direction, exp, timestep, DI, info = val_ds[s["idx"]]
    real_data = real_data.unsqueeze(0).to(device)
    hidden_state = hidden_state.unsqueeze(0).to(device)
    direction = direction.unsqueeze(0).to(device)
    DI = DI.unsqueeze(0).to(device)
    info = info.unsqueeze(0).to(device)

    with torch.no_grad():
        c = model.process_condition(info, hidden_state, direction, DI)
        pred = model.sample_with_ode(c, num_steps=50)
        pred = decode_prediction(model, pred)

    real_eval = torch.clamp(real_data * 0.5 + 0.5, 0.0, 1.0)
    pred_eval = torch.clamp(pred * 0.5 + 0.5, 0.0, 1.0)

    records.append({
        "meta": s,
        "exp": exp,
        "timestep": int(timestep),
        "real_eval": real_eval,
        "pred_eval": pred_eval,
    })

# 7) Batch metrics: PSNR / SSIM / FID
psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
fid_metric = FrechetInceptionDistance(feature=2048, reset_real_features=False, normalize=True).to(device)

with torch.no_grad():
    for r in records:
        psnr_metric.update(r["pred_eval"], r["real_eval"])
        ssim_metric.update(r["pred_eval"], r["real_eval"])
        fid_metric.update(r["real_eval"], real=True)
        fid_metric.update(r["pred_eval"], real=False)

    psnr_val = psnr_metric.compute().item()
    ssim_val = ssim_metric.compute().item()
    fid_val = fid_metric.compute().item()

print(f"Batch PSNR: {psnr_val:.4f}")
print(f"Batch SSIM: {ssim_val:.4f}")
print(f"Batch FID : {fid_val:.4f}")

# 8) Plot 4 rows x 2 cols (GT | Inference)
fig, axes = plt.subplots(len(records), 2, figsize=(8, 3.2 * len(records)))
if len(records) == 1:
    axes = np.array([axes])

for i, r in enumerate(records):
    real_vis = r["real_eval"][0].permute(1, 2, 0).cpu().numpy()
    pred_vis = r["pred_eval"][0].permute(1, 2, 0).cpu().numpy()
    meta = r["meta"]

    axes[i, 0].imshow(real_vis)
    axes[i, 0].set_title(
        f"GT | target_DI={meta['target_di']:.2f} actual_DI={meta['actual_di']:.3f}"
    )
    axes[i, 0].axis("off")

    axes[i, 1].imshow(pred_vis)
    axes[i, 1].set_title(f"Inference | {meta['filename']}")
    axes[i, 1].axis("off")

fig.suptitle(
    f"CFM DI-spread Validation | PSNR={psnr_val:.2f}, SSIM={ssim_val:.4f}, FID={fid_val:.2f}"
)
plt.tight_layout()
plt.show()

save_path = os.path.join(run_dir, "gt_vs_inference_di_spread.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved figure to:", save_path)

In [ ]:
# [Cell 14] Interference-masked DI evaluation (Grounding DINO + loaded DINOv3)
# Prerequisites:
# - Run Cell 13 first; it creates `records` containing generated `pred_eval` images.
# - Load DINOv3 elsewhere in this notebook/session as `dinov3_model` and
#   `dinov3_processor`. This cell deliberately does not load a DINOv2 model.
# - Provide a trained DamageIndexMLP checkpoint. An untrained MLP cannot
#   produce meaningful DI predictions.

import os
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from src.model.ThirdPartyDIEvaluator import InterferenceMaskedDIEvaluator

assert "records" in globals() and records, "Run Cell 13 before running this cell."
assert "dinov3_model" in globals(), "Load DINOv3 first and assign it to `dinov3_model`."
assert "dinov3_processor" in globals(), "Load the matching processor as `dinov3_processor`."

# Update to the checkpoint trained for DamageIndexMLP(DINOv3 pooled features).
DI_MLP_CHECKPOINT = "/content/di_mlp_dinov3.ckpt"
INTERFERENCE_PROMPT = "metal screw. steel rebar."
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

assert os.path.isfile(DI_MLP_CHECKPOINT), (
    "A trained DI MLP checkpoint is required for meaningful DI evaluation. "
    f"Set DI_MLP_CHECKPOINT to a valid file; not found: {DI_MLP_CHECKPOINT}"
)

# The evaluator reuses the already-loaded DINOv3 model and processor.
di_evaluator = InterferenceMaskedDIEvaluator(
    dino=dinov3_model,
    dino_processor=dinov3_processor,
    box_threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD,
    freeze_dino=True,
).to(device)

# Supported checkpoint formats:
# 1) a direct DamageIndexMLP state_dict
# 2) {"di_predictor_state_dict": ...}
# 3) a Lightning state_dict with keys prefixed by "di_predictor."
checkpoint = torch.load(DI_MLP_CHECKPOINT, map_location=device, weights_only=False)
mlp_state = checkpoint.get("di_predictor_state_dict", checkpoint.get("state_dict", checkpoint))
if any(key.startswith("di_predictor.") for key in mlp_state):
    mlp_state = {
        key.removeprefix("di_predictor."): value
        for key, value in mlp_state.items()
        if key.startswith("di_predictor.")
    }
load_result = di_evaluator.di_predictor.load_state_dict(mlp_state, strict=True)
print("Loaded DI MLP checkpoint:", DI_MLP_CHECKPOINT)
print("DI MLP load status:", load_result)
di_evaluator.eval()

# Cell 13 stores generated images in [0, 1], so use input_range="zero_one".
generated_batch = torch.cat([record["pred_eval"] for record in records], dim=0).to(device)
with torch.no_grad():
    di_result = di_evaluator(
        generated_batch,
        text_prompt=INTERFERENCE_PROMPT,
        input_range="zero_one",
        return_details=True,
    )

predicted_di = di_result["predicted_di"].squeeze(1).detach().cpu()
patch_masks = di_result["interference_patch_mask"].detach().cpu()
boxes_per_image = [boxes.detach().cpu() for boxes in di_result["boxes_xyxy"]]

print("\nInterference-masked DI results:")
for index, (record, prediction, patch_mask, boxes) in enumerate(
    zip(records, predicted_di, patch_masks, boxes_per_image)
):
    actual_di = record["meta"]["actual_di"]
    print(
        f"sample={index} actual_DI={actual_di:.4f} predicted_DI={prediction.item():.4f} "
        f"boxes={len(boxes)} masked_patches={int(patch_mask.sum())}/{patch_mask.numel()}"
    )

# Visual audit: red boxes are regions excluded from DINOv3 feature pooling.
fig, axes = plt.subplots(1, len(records), figsize=(4.5 * len(records), 4.5))
if len(records) == 1:
    axes = [axes]

for index, (axis, record, prediction, boxes) in enumerate(
    zip(axes, records, predicted_di, boxes_per_image)
):
    image = record["pred_eval"][0].permute(1, 2, 0).detach().cpu().numpy()
    axis.imshow(image)
    for x1, y1, x2, y2 in boxes.tolist():
        axis.add_patch(
            Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor="red", linewidth=2,
            )
        )
    axis.set_title(
        f"actual={record['meta']['actual_di']:.3f}\npredicted={prediction.item():.3f}"
    )
    axis.axis("off")

fig.suptitle("Grounding DINO interference boxes excluded from DINOv3 DI features")
plt.tight_layout()
plt.show()

# Keep results available for later metrics/export.
di_evaluation_results = {
    "predicted_di": predicted_di,
    "boxes_xyxy": boxes_per_image,
    "interference_patch_mask": patch_masks,
}
print("Saved in-memory results as `di_evaluation_results`.")

In [ ]:
# [Cell 15] Train DI-MLP + GT visual dashboard (run only Cells 1 → 2 → 4 → 15)
# This cell does NOT require CFM training, a CFM checkpoint, or Cells 8/9/12/13/14.
# It trains the DI MLP on GT TRAIN images, validates on GT VAL images, then
# renders one dashboard per validation column. Grounding DINO + DINOv3 are frozen.

import json
import os
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision.transforms.functional import resize, to_tensor
from transformers import AutoImageProcessor, AutoModel

from scripts.visual_debug_dashboard import run_visual_dashboard
from src.model.ThirdPartyDIEvaluator import InterferenceMaskedDIEvaluator

# Model and detector settings.
DINOv3_MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"
INTERFERENCE_PROMPT = "metal screw. steel rebar."
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

# Feature extraction settings. Larger batches use less time but more GPU memory.
FEATURE_IMAGE_SIZE = 518
FEATURE_BATCH_SIZE = 4
FEATURE_CACHE_PATH = "/content/di_mlp_dinov3_features.pt"
REBUILD_FEATURE_CACHE = False

# DI MLP training settings. The final MLP checkpoint is saved for later reuse.
DI_MLP_EPOCHS = 100
DI_MLP_BATCH_SIZE = 32
DI_MLP_LR = 1e-3
DI_MLP_CHECKPOINT = "/content/di_mlp_dinov3.ckpt"


class GTPatternDashboardDataset(Dataset):
    """GT RC images and labels only; no CFM model or extractor is required."""

    def __init__(self, root="Data", split="val"):
        image_list_path = os.path.join(root, "DamagePattern", "id_list.json")
        label_path = os.path.join(root, "DamagePattern", "label.csv")
        self.images_dir = os.path.join(root, "DamagePattern", "images")
        assert os.path.isfile(image_list_path), f"Missing split file: {image_list_path}"
        assert os.path.isfile(label_path), f"Missing label file: {label_path}"

        image_list = json.load(open(image_list_path, "r"))[split]
        self.label = pd.read_csv(label_path)
        self.label = self.label[self.label["filename"].isin(image_list)].reset_index(drop=True)
        assert not self.label.empty, f"No labelled images found for split={split!r}."

    def __len__(self):
        return len(self.label)

    def __getitem__(self, index):
        row = self.label.iloc[index]
        image_path = os.path.join(self.images_dir, row["filename"])
        with Image.open(image_path) as image:
            image = image.convert("RGB").copy()
        return {
            "image": image,
            "column_id": row["exp"],  # `exp` is the project column identifier.
            "DI": float(row["DI"]),
        }


def cache_clean_features(dataset, split_name):
    """Run frozen detector+DINO once; cache clean pooled features for MLP training."""
    features, labels = [], []
    print(f"Extracting masked DINOv3 features for {split_name}: {len(dataset)} images")

    for start in range(0, len(dataset), FEATURE_BATCH_SIZE):
        samples = [dataset[i] for i in range(start, min(start + FEATURE_BATCH_SIZE, len(dataset)))]
        # A fixed size permits batching while preserving the detector→patch mapping.
        images = torch.stack([
            to_tensor(resize(sample["image"], [FEATURE_IMAGE_SIZE, FEATURE_IMAGE_SIZE]))
            for sample in samples
        ]).to(device)
        labels_batch = torch.tensor([sample["DI"] for sample in samples], dtype=torch.float32)

        with torch.no_grad():
            details = di_evaluator(
                images,
                text_prompt=INTERFERENCE_PROMPT,
                input_range="zero_one",
                return_details=True,
            )
        features.append(details["clean_features"].detach().cpu())
        labels.append(labels_batch)
        if start == 0 or (start + len(samples)) % (FEATURE_BATCH_SIZE * 25) == 0:
            print(f"  {min(start + len(samples), len(dataset))}/{len(dataset)}")

    return torch.cat(features), torch.cat(labels).unsqueeze(1)


def evaluate_di_mlp(features, labels):
    """Return predictions and scalar validation metrics for cached features."""
    di_evaluator.di_predictor.eval()
    with torch.no_grad():
        predictions = di_evaluator.di_predictor(features.to(device)).cpu()
    mae = F.l1_loss(predictions, labels).item()
    rmse = F.mse_loss(predictions, labels).sqrt().item()
    total_variance = ((labels - labels.mean()) ** 2).sum().clamp_min(1e-12)
    r2 = 1.0 - (((labels - predictions) ** 2).sum() / total_variance).item()
    return predictions, {"mae": mae, "rmse": rmse, "r2": r2}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Loading DINOv3:", DINOv3_MODEL_ID)
dinov3_processor = AutoImageProcessor.from_pretrained(DINOv3_MODEL_ID)
dinov3_model = AutoModel.from_pretrained(DINOv3_MODEL_ID).to(device).eval()

di_evaluator = InterferenceMaskedDIEvaluator(
    dino=dinov3_model,
    dino_processor=dinov3_processor,
    box_threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD,
    freeze_dino=True,
).to(device).eval()

train_ds = GTPatternDashboardDataset(root="Data", split="train")
val_ds = GTPatternDashboardDataset(root="Data", split="val")
print(f"Train: {len(train_ds)} images / {train_ds.label['exp'].nunique()} columns")
print(f"Val  : {len(val_ds)} images / {val_ds.label['exp'].nunique()} columns")

# Feature caching avoids repeatedly running Grounding DINO and DINOv3 every epoch.
if os.path.isfile(FEATURE_CACHE_PATH) and not REBUILD_FEATURE_CACHE:
    feature_cache = torch.load(FEATURE_CACHE_PATH, map_location="cpu", weights_only=False)
    train_features, train_labels = feature_cache["train_features"], feature_cache["train_labels"]
    val_features, val_labels = feature_cache["val_features"], feature_cache["val_labels"]
    print("Loaded cached clean DINOv3 features:", FEATURE_CACHE_PATH)
else:
    train_features, train_labels = cache_clean_features(train_ds, "train")
    val_features, val_labels = cache_clean_features(val_ds, "val")
    torch.save(
        {
            "train_features": train_features,
            "train_labels": train_labels,
            "val_features": val_features,
            "val_labels": val_labels,
            "dino_model_id": DINOv3_MODEL_ID,
            "prompt": INTERFERENCE_PROMPT,
        },
        FEATURE_CACHE_PATH,
    )
    print("Saved clean DINOv3 feature cache:", FEATURE_CACHE_PATH)

# Train only the MLP; Grounding DINO and DINOv3 remain frozen feature extractors.
di_evaluator.di_predictor.train()
optimizer = torch.optim.AdamW(di_evaluator.di_predictor.parameters(), lr=DI_MLP_LR, weight_decay=1e-4)
train_loader = DataLoader(TensorDataset(train_features, train_labels), batch_size=DI_MLP_BATCH_SIZE, shuffle=True)

for epoch in range(1, DI_MLP_EPOCHS + 1):
    di_evaluator.di_predictor.train()  # evaluate_di_mlp() switches it to eval mode.
    epoch_losses = []
    for feature_batch, label_batch in train_loader:
        prediction = di_evaluator.di_predictor(feature_batch.to(device))
        loss = F.mse_loss(prediction, label_batch.to(device))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.detach())

    if epoch == 1 or epoch % 10 == 0 or epoch == DI_MLP_EPOCHS:
        _, val_metrics = evaluate_di_mlp(val_features, val_labels)
        print(
            f"Epoch {epoch:03d}/{DI_MLP_EPOCHS} "
            f"train_MSE={torch.stack(epoch_losses).mean().item():.5f} "
            f"val_MAE={val_metrics['mae']:.4f} val_RMSE={val_metrics['rmse']:.4f} val_R2={val_metrics['r2']:.4f}"
        )

val_predictions, val_metrics = evaluate_di_mlp(val_features, val_labels)
di_evaluator.di_predictor_is_trained = True
torch.save(
    {
        "di_predictor_state_dict": di_evaluator.di_predictor.state_dict(),
        "dino_model_id": DINOv3_MODEL_ID,
        "prompt": INTERFERENCE_PROMPT,
        "validation_metrics": val_metrics,
    },
    DI_MLP_CHECKPOINT,
)
print("\nSaved trained DI MLP:", DI_MLP_CHECKPOINT)
print("Final validation metrics:", val_metrics)

# Plot 1: GT DI from label.csv. Plot 4: DI from the trained masked-feature MLP.
# Each dashboard: Original | Detection | Patch Mask Alignment | Masked View.
dashboard_figures = run_visual_dashboard(val_ds, di_evaluator)
print(f"Generated {len(dashboard_figures)} dashboard figure(s): {list(dashboard_figures)}")